# 260902
[machine learning chapter06-2]

---

## 1. Today's Goal

- 배운개념 정리 및 이해
- 실습코드 플로우 이해


---

## 2. Key Concepts

---

## 3. Practice 실습 코드

실습 전 의존성 패키지 설치
- uv add transformers torch sentencepiece pandas matplotlib accelerate ipykernel

In [ ]:
import platform 
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 가속 연산 디바이스 자동 설정
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)
print("현재 사용 중 디바이스:", device)

# 사전 학습된 인코더 모델 및 토크나이저 로드
model_name = "rkdaldus/ko-sent5-classification"
tokenizer = AutoTokenizer.from_pretrained("monologg/kobert", trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

# 분석 테스트 변수 정의
text = "이번 시험을 위해 밤을 새우며 정말 열심히 준비했는데 생각보다 점수가 너무 낮게 나와서 속상하고 눈물이 나네."

# 분석 텍스트 토큰화 및 PyTorch tensor로 변환 - encoding
encoded_input = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

# 로드된 모델을 통한 감정별 logit 추출
with torch.no_grad(): # --> 예측 중일때는 가중치를 업데이트 하지 않으므로 no_grad()
    model_output = model(**encoded_input) # ** -> dictionary의 key-valuefmf **keyargs 로 펼치기
    logits = model_output.logits

# softmax로 5가지 감정별 확률 계산
calculated_possibilities = torch.nn.functional.softmax(logits, dim=-1).squeeze().tolist()

# 감정 레이블 매핑
emotion_labels = ["anger", "anxiousness", "happiness", "calmness", "sadness"]

# 감정 레이블을 데이터프레임으로 변환
emotion_df = pd.DataFrame({
    "Emotion": emotion_labels,
    "Probability": calculated_possibilities
})

print(emotion_df)

현재 사용 중 디바이스: mps


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12361.77it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: rkdaldus/ko-sent5-classification
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


       Emotion  Probability
0        anger     0.239532
1  anxiousness     0.182797
2    happiness     0.141758
3     calmness     0.200832
4      sadness     0.235081


--- 
## 4. Daily Quest

[틀린 문제]
> 트랜스포머의 인코더-디코더 결합 구조가 주로 활용되는 대표적인 자연어 처리 태스크를 한 가지 이상 쓰시오.

- 오답: BART
- 정답: 
- 해설:  인코더-디코더 구조는 입력 문장을 인코더로 맥락화한 뒤 디코더가 새로운 문장을 생성하는 방식이므로, KoBART 실습처럼 문서 요약이나 기계 번역에 주로 사용됩니다.
